In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio import features
from shapely.geometry import shape
from shapely.ops import unary_union
from shapely.validation import make_valid

Paths etc.

In [2]:
GUPPD_PATH = 'urbanspatial-guppd-v1-gpkg.gpkg'  # GUPPD polygons
GUPPD_LAYER = 'urbanspatial_guppd_v1_polygons'  # urban centre (30), dense (23) and semi-dense (22) cells

SMOD_RASTER_PATH = 'GHS_SMOD_POP2015_GLOBE_R2019A_54009_1K_V2_0.tif'  # GHS-SMOD raster to extract peri-urban class

COUNTRIES_PATH = 'Files/base_data/cartographic/ee/ee_r264_correspondence.gpkg'  # country boundaries

OUT_GPKG = 'urban_suburbs_denserural_boundaries_2019.gpkg'

EXTRA_SMOD_CLASSES = [21,13]

PRIORITY = ['30', '23', '22', '21', '13']  # priority ranking for overlapping polygons



Helper functions

In [3]:
def filter_polygons_only(geometry):
    """
    Filters out invalid geometries.
    """
    if geometry.is_empty or geometry.geom_type in ('Polygon', 'MultiPolygon'):
        return geometry
    parts = [p for p in getattr(geometry, 'geoms', []) if p.geom_type in ('Polygon', 'MultiPolygon')]  # extract only the polygon parts from a collection
    return unary_union(parts) if parts else geometry.__class__()  # merge'em together or return empty geometry if none exists


def find_overlapping_polygons(geometries):
    """
    Finds all pairs of polygons that physically overlap each other.
    Calculates the exact overlap area in square kilometers.
    """
    # create a spatial index to find intersecting bounding boxes
    gdf_left = gpd.GeoDataFrame(geometry=geometries.reset_index(drop=True), crs=geometries.crs)
    pairs = gpd.sjoin(gdf_left, gdf_left, how='inner', predicate='intersects')
    
    pairs = pairs[pairs.index < pairs['index_right']]  # avoid duplication
    
    # extract actual geometries for the overlapping pairs
    geom_1 = gdf_left.geometry.loc[pairs.index].reset_index(drop=True)
    geom_2 = gdf_left.geometry.loc[pairs['index_right']].reset_index(drop=True)
    
    overlap_area_km2 = geom_1.intersection(geom_2, align=False).to_crs('ESRI:54009').area / 1e6  # intersection area
    overlaps = pd.DataFrame({'poly_1': pairs.index.values, 'poly_2': pairs['index_right'].values, 'overlap_km2': overlap_area_km2.values})
    
    return overlaps[overlaps['overlap_km2'] > 1e-6].reset_index(drop=True)


def resolve_overlapping_polygons(gdf, overlap_pairs):
    """
    Iterates through overlapping pairs and cuts the lower-priority polygon to avoid double counting.
    Removes any polygon completely swallowed by a higher-priority one.
    """
    geometries = list(gdf.geometry)
    
    priorities = [PRIORITY.index(level) for level in gdf['smod_level']]  # convert string levels ('30' etc.) into numerical
    higher_priority_map = {}
    
    # dictionary mapping each lower-priority polygon to higher-priority polygons cutting into it
    for p1, p2 in zip(overlap_pairs['poly_1'], overlap_pairs['poly_2']):
        keep, cut = (p1, p2) if (priorities[p1], p1) < (priorities[p2], p2) else (p2, p1)
        higher_priority_map.setdefault(cut, []).append(keep)
        
    # perform cuts starting with highest priority polygons
    for cut_idx in sorted(higher_priority_map, key=lambda c: (priorities[c], c)):
        new_geom = geometries[cut_idx].difference(unary_union([geometries[k] for k in higher_priority_map[cut_idx]]))
                # subtract union of all overlapping higher-priority polygons from this one
        geometries[cut_idx] = filter_polygons_only(new_geom if new_geom.is_valid else make_valid(new_geom))  # repair geom
        
    # update df with newly cut geometries
    resolved_gdf = gdf.reset_index(drop=True).copy()
    resolved_gdf['geometry'] = geometries
    
    return resolved_gdf[~resolved_gdf.geometry.is_empty]  # drop completely erased polygons

GUPPD

In [4]:
guppd_gdf = gpd.read_file(GUPPD_PATH, layer=GUPPD_LAYER)

# mapping dictionary to fill missing country names based on most common name for that ISO code
iso_to_name = guppd_gdf.dropna(subset=['CNTRY_NAME']).groupby('ISO')['CNTRY_NAME'].agg(lambda s: s.mode().iloc[0])

# standardize GUPPD df format for a clean merge later
guppd_clean = gpd.GeoDataFrame({
    'id': guppd_gdf['SMOD_ID'].astype(str),
    'smod_level': guppd_gdf['SMOD_LEVEL'].astype(str),
    'country': guppd_gdf['CNTRY_NAME'].fillna(guppd_gdf['ISO'].map(iso_to_name)).fillna('Unknown'),
    'iso3': guppd_gdf['ISO'].astype(str).str.strip().str.upper(),
    'region_name': guppd_gdf['CIESIN_NAME'],
    'geometry': guppd_gdf.geometry,
}, crs=guppd_gdf.crs)


Vectorize SMOD class 21

In [ ]:
# open the raster file to extract class 21 boundaries
with rasterio.open(SMOD_RASTER_PATH) as src:
    smod_raster = src.read(1)
    smod_crs = src.crs
    smod_transform = src.transform

extra_polygons = []
for smod_class in EXTRA_SMOD_CLASSES:
    class_mask = smod_raster == smod_class  # boolean mask pixels of class 21
    # extract contiguous pixel blobs into vector polygons
    polygons = [
        shape(geom) 
        for geom, _ in features.shapes(
            class_mask.astype('uint8'), 
            mask=class_mask, 
            transform=smod_transform, 
            connectivity=4  # ensures diagonals are not counted as continuous settlements
        )
    ]
    extra_polygons.append(
        gpd.GeoDataFrame(
            {'smod_level': [str(smod_class)] * len(polygons)},
            geometry=polygons, crs=smod_crs
        )
    )

# create a gdf from newly extracted polygons
smod_gdf = gpd.GeoDataFrame(
    pd.concat(extra_polygons, ignore_index=True),
    crs=smod_crs
).to_crs(guppd_clean.crs)


# Assign countries to new SMOD polygons.

countries_gdf = gpd.read_file(COUNTRIES_PATH)[['iso3_r250_label', 'geometry']].to_crs(guppd_clean.crs)  # load country boundaries
points_gdf = gpd.GeoDataFrame(geometry=smod_gdf.geometry.representative_point(), crs=smod_gdf.crs)  # generate a representative point inside each new polygon for spatial join
country_hits = gpd.sjoin(points_gdf, countries_gdf, how='left', predicate='intersects')  # which country boundary contains each polygon's representative point

# ensure no duplicate rows if a point accidentally hits a border perfectly
country_hits = country_hits[~country_hits.index.duplicated()]
smod_gdf['iso3'] = country_hits['iso3_r250_label'].reindex(smod_gdf.index)

# handle orphaned polygons like coastal or border glitches
missing_iso = smod_gdf['iso3'].isna()
countries_ea = countries_gdf.to_crs('ESRI:54009')

#for any point that didn't intersect a country measure distance to all countries and assign it to the nearest
for idx, point in points_gdf.loc[missing_iso].to_crs('ESRI:54009').geometry.items():
    smod_gdf.loc[idx, 'iso3'] = countries_gdf.loc[countries_ea.distance(point).idxmin(), 'iso3_r250_label']

# assign formatted IDs, map ISO3 back to standard country names
smod_gdf['id'] = [f'{level}_{i:07d}' for i,level in enumerate(smod_gdf['smod_level'])]
smod_gdf['country'] = smod_gdf['iso3'].map(iso_to_name).fillna(smod_gdf['iso3'])
smod_gdf['region_name'] = ''



Combine and resolve overlaps

In [6]:
combined_gdf = gpd.GeoDataFrame(
    pd.concat([guppd_clean, smod_gdf[guppd_clean.columns]], ignore_index=True), crs=guppd_clean.crs
)

# filter out geometries with infinite or NaN coordinates in their bb
valid_bounds = np.isfinite(combined_gdf.bounds).all(axis=1)
combined_gdf = combined_gdf[valid_bounds].reset_index(drop=True)

invalid_geoms = ~combined_gdf.geometry.is_valid
if invalid_geoms.any():
    combined_gdf.loc[invalid_geoms, 'geometry'] = combined_gdf.loc[invalid_geoms, 'geometry'].apply(make_valid)

combined_gdf['geometry'] = combined_gdf['geometry'].apply(filter_polygons_only)
combined_gdf = combined_gdf[~combined_gdf.geometry.is_empty].reset_index(drop=True)

# resolve priorities where polygons overlap
overlap_pairs = find_overlapping_polygons(combined_gdf.geometry)
final_gdf = resolve_overlapping_polygons(combined_gdf, overlap_pairs)
final_gdf['area_km2'] = final_gdf.to_crs('ESRI:54009').geometry.area / 1e6


Export

In [7]:
if os.path.exists(OUT_GPKG):
    os.remove(OUT_GPKG)
final_gdf[['id', 'smod_level', 'country', 'iso3', 'region_name', 'area_km2', 'geometry']] \
.to_file(OUT_GPKG, driver='GPKG', layer='urban_suburbs_denserural_boundaries_2019')


